# ERCOT Houston Hub — EDA & Volatility Modeling Plan

This notebook covers:
1. **Setup** — load processed data, define constants
2. **Target EDA** — distribution and time-series structure of RTM price volatility
3. **Feature EDA** — key predictors (net load, wind error, forecast uncertainty, ancillary prices)
4. **Regime analysis** — extreme events (Winter Storm Uri, summer peaks, autocorrelation)
5. **Modeling roadmap** — recommended models, features, and validation strategy

**Data range:** 2017-07-01 → 2025-12-31 (hourly, ~74,000 rows).  
**2026 data is held out as out-of-sample test set — do not load it here.**

## 1. Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats

PROCESSED_ROOT = Path('data/processed/ercot')

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
sns.set_style('whitegrid')

# Spike threshold — adjust after EDA below
SPIKE_THRESHOLD = 500   # $/MWh

print('Imports OK')

In [ ]:
df = pd.read_parquet(PROCESSED_ROOT / 'ercot_combined.parquet')
df['ts_utc'] = pd.to_datetime(df['ts_utc'])
df = df.sort_values('ts_utc').reset_index(drop=True)

print(f'Rows: {len(df):,}  |  Columns: {len(df.columns)}')
print(f'Range: {df.ts_utc.min()} -> {df.ts_utc.max()}')
print(f'\nColumns:\n{list(df.columns)}')

## 2. Target Variable: RTM Price Volatility

**Primary target:** `rtm_price_std_hb_houston` — std of the 4 intra-hour 15-min RTM prices.
High std = the price moved a lot within that hour = realized intra-hour volatility.
The distribution is heavily right-skewed; use `log1p` transform for regression.

**Secondary target (classification):** `rtm_price_mean_hb_houston > SPIKE_THRESHOLD`.

In [ ]:
# Distribution of RTM price std
_vol = df['rtm_price_std_hb_houston'].dropna()
_log_vol = np.log1p(_vol)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(_vol.clip(upper=500), bins=100, color='steelblue', alpha=0.8)
axes[0].set_title('RTM price std (raw, clipped <=500 $/MWh)')
axes[0].set_xlabel('$/MWh'); axes[0].set_ylabel('count')

axes[1].hist(_log_vol, bins=80, color='darkorange', alpha=0.8)
axes[1].set_title('log1p(RTM price std)')
axes[1].set_xlabel('log1p($/MWh)')

stats.probplot(_log_vol, dist='norm', plot=axes[2])
axes[2].set_title('Q-Q plot of log1p(RTM price std)')

plt.tight_layout()
plt.show()

print(f'Raw:    mean={_vol.mean():.1f}  median={_vol.median():.1f}  '
      f'p95={_vol.quantile(.95):.1f}  p99={_vol.quantile(.99):.1f}  '
      f'max={_vol.max():.1f}')
print(f'Skewness (raw): {_vol.skew():.2f}  |  skewness (log1p): {_log_vol.skew():.2f}')

In [ ]:
# Full time series of RTM volatility (daily mean)
_daily_vol = (
    df.set_index('ts_utc')['rtm_price_std_hb_houston']
      .resample('D').mean()
)

fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(_daily_vol.index, _daily_vol.values, lw=0.5, color='steelblue', alpha=0.7)
ax.axvline(pd.Timestamp('2021-02-10'), color='red', lw=1.5, linestyle='--', label='Uri (Feb 2021)')
ax.set_title('Daily mean RTM intra-hour price std — HB_HOUSTON  (2017-07 to 2025-12)')
ax.set_ylabel('$/MWh'); ax.legend(fontsize=9)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Diurnal and seasonal volatility patterns
_months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df.groupby('hour')['rtm_price_std_hb_houston'].median().plot(
    ax=axes[0], marker='o', ms=4, lw=1.5, color='steelblue')
axes[0].set_title('Median RTM price std by hour of day (CST/UTC-6)')
axes[0].set_xlabel('CST hour'); axes[0].set_ylabel('$/MWh')
axes[0].set_xticks(range(0, 24, 2))

df.groupby('month')['rtm_price_std_hb_houston'].median().plot(
    kind='bar', ax=axes[1], color='darkorange', alpha=0.85, width=0.8)
axes[1].set_title('Median RTM price std by month')
axes[1].set_xticklabels(_months, rotation=30, ha='right')
axes[1].set_ylabel('$/MWh')

plt.tight_layout()
plt.show()

In [ ]:
# Spike rate and threshold selection
_rtm = df['rtm_price_mean_hb_houston'].dropna()
for thr in [100, 200, 500, 1000]:
    n = (_rtm > thr).sum()
    print(f'  RTM mean > {thr:>5} $/MWh:  {n:>6,} hours  ({100*n/len(_rtm):.3f}%)')

print(f'\np99  = {_rtm.quantile(.99):.1f} $/MWh')
print(f'p999 = {_rtm.quantile(.999):.1f} $/MWh')
print(f'\nSPIKE_THRESHOLD = {SPIKE_THRESHOLD} $/MWh')
print(f'  -> {(_rtm > SPIKE_THRESHOLD).sum():,} spike hours  '
      f'({100*(_rtm > SPIKE_THRESHOLD).mean():.3f}%)')

## 3. Feature EDA

### 3.1 Net Load — primary driver of price volatility

`net_load_mw = load_total - total_irr_mw`  
When net load approaches the top of the supply stack, marginal costs rise steeply.
This relationship is nonlinear — include `net_load_mw^2` as a feature.

In [ ]:
# Net load vs. RTM volatility
_sub = df[['net_load_mw', 'rtm_price_std_hb_houston']].dropna()
_sub = _sub.copy()
_sub['nl_bin'] = pd.qcut(_sub['net_load_mw'], q=20)
_binned = _sub.groupby('nl_bin', observed=True)['rtm_price_std_hb_houston'].median()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

idx = _sub.sample(min(5000, len(_sub)), random_state=42).index
axes[0].scatter(_sub.loc[idx, 'net_load_mw'],
                np.log1p(_sub.loc[idx, 'rtm_price_std_hb_houston']),
                alpha=0.1, s=3, color='steelblue')
axes[0].set_xlabel('Net load (MW)'); axes[0].set_ylabel('log1p(RTM price std)')
axes[0].set_title('Net load vs. log-volatility (5k sample)')

_binned.plot(ax=axes[1], marker='o', ms=4, lw=1.5, color='darkorange')
axes[1].set_title('Median RTM price std by net load quantile bin')
axes[1].set_xlabel('Net load quantile bin'); axes[1].set_ylabel('$/MWh')
axes[1].set_xticklabels([])

plt.tight_layout()
plt.show()

corr = np.corrcoef(_sub['net_load_mw'], np.log1p(_sub['rtm_price_std_hb_houston']))[0,1]
print(f'Correlation (net_load_mw vs log-vol): {corr:.3f}')

### 3.2 Wind Forecast Error — proximate cause of RTM spikes

`wind_error_system = system_wide_gen - stwpf_system_wide`  
Negative = wind underperformed forecast -> real-time must dispatch expensive backup.  
The relationship is asymmetric: underperformance hurts more than overperformance helps.

In [ ]:
# Wind error vs. RTM volatility
_sub2 = df[['wind_error_system', 'rtm_price_std_hb_houston']].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

idx2 = _sub2.sample(min(5000, len(_sub2)), random_state=42).index
axes[0].scatter(_sub2.loc[idx2, 'wind_error_system'],
                np.log1p(_sub2.loc[idx2, 'rtm_price_std_hb_houston']),
                alpha=0.1, s=3, color='forestgreen')
axes[0].axvline(0, color='red', lw=0.8, linestyle='--')
axes[0].set_xlabel('Wind error (actual - STWPF, MW)')
axes[0].set_ylabel('log1p(RTM price std)')
axes[0].set_title('Wind forecast error vs. log-volatility (5k sample)')

_neg = _sub2[_sub2['wind_error_system'] < 0]['rtm_price_std_hb_houston']
_pos = _sub2[_sub2['wind_error_system'] >= 0]['rtm_price_std_hb_houston']
axes[1].boxplot([_neg.clip(upper=200), _pos.clip(upper=200)],
                labels=['Wind < STWPF\n(under)', 'Wind >= STWPF\n(over)'],
                notch=True)
axes[1].set_title('RTM price std: wind under- vs. over-performance')
axes[1].set_ylabel('$/MWh (clipped <=200)')

plt.tight_layout()
plt.show()

print(f'Median vol when wind < STWPF : {_neg.median():.2f} $/MWh  (n={len(_neg):,})')
print(f'Median vol when wind >= STWPF: {_pos.median():.2f} $/MWh  (n={len(_pos):,})')

### 3.3 Forecast Revision Std — forward-looking uncertainty signal

`fc_system_total_std_48h` / `wf_stwpf_system_wide_std_48h` measure disagreement across
the D+1…D+7 forecast sequence.  High revision std = the market was uncertain about this
delivery hour days in advance — a genuine pre-delivery signal of volatility risk.

In [ ]:
# Forecast revision std vs. RTM volatility
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, fcol, label, color in [
    (axes[0], 'fc_system_total_std_48h',      'Load forecast rev std (MW)',   'steelblue'),
    (axes[1], 'wf_stwpf_system_wide_std_48h', 'Wind forecast rev std (MW)',   'forestgreen'),
]:
    if fcol not in df.columns:
        ax.text(0.3, 0.5, f'{fcol}\nnot found', transform=ax.transAxes)
        continue
    _s = df[[fcol, 'rtm_price_std_hb_houston']].dropna()
    idx = _s.sample(min(5000, len(_s)), random_state=42).index
    ax.scatter(_s.loc[idx, fcol],
               np.log1p(_s.loc[idx, 'rtm_price_std_hb_houston']),
               alpha=0.15, s=3, color=color)
    ax.set_xlabel(label)
    ax.set_ylabel('log1p(RTM price std)')
    ax.set_title(f'{label} vs. log-volatility')
    corr = np.corrcoef(_s[fcol], np.log1p(_s['rtm_price_std_hb_houston']))[0,1]
    ax.text(0.05, 0.92, f'corr={corr:.3f}', transform=ax.transAxes, fontsize=9)

plt.tight_layout()
plt.show()

### 3.4 DAM Price & DAM-RTM Spread

The day-ahead price reflects the market's prior expectation of scarcity.
`dam_rtm_spread = dam_price - rtm_mean`: negative when real-time was tighter than expected.
For D+1 forecasting, use the **previous day's** DAM price (known before delivery).

In [ ]:
# DAM price and spread vs. RTM volatility
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, col, label, color in [
    (axes[0], 'dam_price_houston',      'DAM price HB_HOUSTON ($/MWh)',      'navy'),
    (axes[1], 'dam_rtm_spread_houston', 'DAM-RTM spread HB_HOUSTON ($/MWh)', 'crimson'),
]:
    if col not in df.columns:
        ax.text(0.3, 0.5, f'{col}\nnot found', transform=ax.transAxes)
        continue
    _s = df[[col, 'rtm_price_std_hb_houston']].dropna()
    _clip = _s[col].clip(-200, 500)
    idx = _s.sample(min(5000, len(_s)), random_state=42).index
    ax.scatter(_clip.loc[idx],
               np.log1p(_s.loc[idx, 'rtm_price_std_hb_houston']),
               alpha=0.12, s=3, color=color)
    ax.set_xlabel(label)
    ax.set_ylabel('log1p(RTM price std)')
    ax.set_title(f'{label} vs. log-volatility')
    corr = np.corrcoef(_s[col], np.log1p(_s['rtm_price_std_hb_houston']))[0,1]
    ax.text(0.05, 0.92, f'corr={corr:.3f}', transform=ax.transAxes, fontsize=9)

plt.tight_layout()
plt.show()

### 3.5 Ancillary Service Prices — leading indicators of reserve tightness

When REGUP and RRS prices spike in the DAM, the system operator expected a tight reserve margin
the next day — a leading indicator of RTM volatility.

In [ ]:
# Ancillary prices vs. RTM volatility
_anc_cols = [c for c in ['mcpc_rrs','mcpc_regup','mcpc_ecrs','mcpc_regdn','mcpc_nsrs']
             if c in df.columns]

corrs = {}
for c in _anc_cols:
    _s = df[[c, 'rtm_price_std_hb_houston']].dropna()
    corrs[c] = np.corrcoef(_s[c], np.log1p(_s['rtm_price_std_hb_houston']))[0,1]

fig, ax = plt.subplots(figsize=(7, 3))
pd.Series(corrs).sort_values().plot(kind='barh', ax=ax, color='purple', alpha=0.8)
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Correlation of ancillary MCPC prices with log1p(RTM price std)')
ax.set_xlabel('Pearson r')
plt.tight_layout()
plt.show()

## 4. Regime Analysis

### 4.1 Winter Storm Uri (Feb 2021)

In [ ]:
# Winter Storm Uri zoom
_lo, _hi = pd.Timestamp('2021-02-08'), pd.Timestamp('2021-02-21')
_uri = df[(df['ts_utc'] >= _lo) & (df['ts_utc'] <= _hi)].copy()

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
fig.suptitle('Winter Storm Uri — Feb 8-20, 2021', fontsize=13)

axes[0].plot(_uri['ts_utc'], _uri['rtm_price_mean_hb_houston'].clip(upper=10000),
             lw=1.2, color='crimson', label='RTM mean')
axes[0].plot(_uri['ts_utc'], _uri['dam_price_houston'],
             lw=1.2, color='navy', alpha=0.8, label='DAM')
axes[0].set_title('Prices $/MWh (RTM clipped <=10,000)'); axes[0].set_ylabel('$/MWh')
axes[0].legend(fontsize=8)

axes[1].plot(_uri['ts_utc'], _uri['rtm_price_std_hb_houston'].clip(upper=5000),
             lw=1.0, color='darkorange')
axes[1].set_title('RTM intra-hour price std (volatility, clipped <=5,000)'); axes[1].set_ylabel('$/MWh')

axes[2].plot(_uri['ts_utc'], _uri['load_total'], lw=1.0, color='steelblue', label='Total load')
if 'net_load_mw' in _uri.columns:
    axes[2].plot(_uri['ts_utc'], _uri['net_load_mw'], lw=1.0, color='purple',
                 alpha=0.8, label='Net load')
axes[2].set_title('Load and net load (MW)'); axes[2].set_ylabel('MW'); axes[2].legend(fontsize=8)

axes[3].plot(_uri['ts_utc'], _uri['wind_error_system'], lw=0.8, color='forestgreen')
axes[3].axhline(0, color='black', lw=0.6, linestyle='--')
axes[3].set_title('Wind error (actual - STWPF, MW)'); axes[3].set_ylabel('MW')

for ax in axes:
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

### 4.2 High-Volatility Hours — seasonal and diurnal concentration

In [ ]:
# Top-10% most volatile hours: breakdown by month and hour of day
_p90 = df['rtm_price_std_hb_houston'].quantile(0.90)
_hi_vol = df[df['rtm_price_std_hb_houston'] > _p90].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
_months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

_hi_vol.groupby('month').size().plot(
    kind='bar', ax=axes[0], color='crimson', alpha=0.85, width=0.8)
axes[0].set_title(f'High-volatility hours (top 10%, >{_p90:.0f} $/MWh) by month')
axes[0].set_xticklabels(_months, rotation=30, ha='right')
axes[0].set_ylabel('Count')

_hi_vol.groupby('hour').size().plot(
    kind='bar', ax=axes[1], color='darkorange', alpha=0.85, width=0.8)
axes[1].set_title('High-volatility hours by CST hour of day')
axes[1].set_xlabel('CST hour (UTC-6)'); axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f'p90 threshold: {_p90:.1f} $/MWh')
print('High-vol hours by season:')
print(_hi_vol.groupby('season').size().sort_values(ascending=False))

### 4.3 Autocorrelation of Volatility — persistence justifies lag features

In [ ]:
# Autocorrelation of log-volatility at key lags
_lv = df.set_index('ts_utc')['rtm_price_std_hb_houston'].dropna()
_log_lv = np.log1p(_lv)

lags = [1, 2, 3, 6, 12, 24, 48, 72, 168]
acf_vals = {lag: _log_lv.autocorr(lag=lag) for lag in lags}

fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(range(len(lags)), list(acf_vals.values()), color='steelblue', alpha=0.85)
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(range(len(lags)))
ax.set_xticklabels([f'lag {l}h' for l in lags])
ax.set_title('Autocorrelation of log1p(RTM price std) at key lags')
ax.set_ylabel('ACF')
plt.tight_layout()
plt.show()

for lag, val in acf_vals.items():
    print(f'  lag {lag:>3}h: {val:.4f}')

## 5. Modeling Roadmap

### 5.1 Feature Correlation Summary

In [ ]:
# Correlation of candidate features with log-volatility
_feature_cols = [
    'net_load_mw', 'load_total', 're_share', 'total_resource_mw',
    'wind_error_system', 'stwpf_system_wide',
    'dam_price_houston', 'dam_rtm_spread_houston',
    'system_lambda',
    'fc_system_total_std_48h', 'wf_stwpf_system_wide_std_48h',
    'genf_total_resource_mw_std_48h',
    'mcpc_rrs', 'mcpc_regup',
    'hour', 'month',
]
_feature_cols = [c for c in _feature_cols if c in df.columns]

df['_log_vol'] = np.log1p(df['rtm_price_std_hb_houston'])
_corr = (df[_feature_cols + ['_log_vol']]
         .corr()['_log_vol']
         .drop('_log_vol')
         .sort_values())
df.drop(columns=['_log_vol'], inplace=True)

colors = ['steelblue' if v >= 0 else 'crimson' for v in _corr]
fig, ax = plt.subplots(figsize=(8, 6))
_corr.plot(kind='barh', ax=ax, color=colors, alpha=0.85)
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Pearson correlation with log1p(RTM price std)')
ax.set_xlabel('Pearson r')
plt.tight_layout()
plt.show()

### 5.2 Modeling Plan

#### Target variables

| Target | Type | Notes |
|---|---|---|
| `log1p(rtm_price_std_hb_houston)` | Regression | Primary. Log stabilizes right-skewed distribution. Evaluate via MAE/RMSE on original scale (`expm1`). |
| `rtm_price_mean_hb_houston > SPIKE_THRESHOLD` | Binary classification | Secondary. Severe class imbalance — use precision-recall AUC, not accuracy. |

#### Feature tiers

| Tier | Features | Why |
|---|---|---|
| 1 | `net_load_mw`, `net_load_mw^2`, `wind_error_system`, `dam_price_houston` (lag 24h), `total_resource_mw` | Direct physical drivers of price spikes |
| 2 | `fc_system_total_std_48h`, `wf_stwpf_system_wide_std_48h`, `genf_total_resource_mw_std_48h`, `mcpc_rrs`, `mcpc_regup`, `dam_rtm_spread_houston` (lag 24h) | Forward-looking uncertainty + reserve signals |
| 3 | `hour`, `month`, `season`, `is_peak`, `weekday` | Diurnal / seasonal regime |
| Lags | `rtm_price_std` at 1h, 24h, 168h; `wind_error_system` at 1h, 2h | Volatility autocorrelation |

**D+1 leakage rule:** For operational D+1 forecasting, only use features available before delivery:
- DAM prices are published the day before ✓
- D+1 forecast features (`fc_*`, `wf_*`, `genf_*`) are available before delivery ✓
- `wind_error_system` and `rtm_price_mean` are realized — **use as lags only** ✓

#### Models (execution order)

| Step | Model | Purpose |
|---|---|---|
| 1 | Seasonal naive: `rtm_price_std` at same hour, 1 week prior | Baseline to beat |
| 2 | Ridge Regression on `log1p(vol)` with polynomial `net_load_mw` | Transparent linear baseline |
| 3 | SARIMA(1,0,1)(1,0,1,24)-X | Time-series baseline; validates that tabular features add signal beyond autocorrelation |
| 4 | **XGBoost** on `log1p(vol)` with time-series CV | Primary model; handles nonlinearity and extreme hours naturally |
| 5 | Random Forest with `class_weight='balanced'` | Spike classification (secondary target) |

#### Validation strategy — expanding window (never random split)

```
Fold 1: train 2017-07 -> 2021-12,  validate 2022
Fold 2: train 2017-07 -> 2022-12,  validate 2023
Fold 3: train 2017-07 -> 2023-12,  validate 2024
Fold 4: train 2017-07 -> 2024-12,  validate 2025  <- final model selection
Out-of-sample test: 2026 (held out, never touched)
```

Use `TimeSeriesSplit(n_splits=4, gap=24)` to avoid same-day leakage between train and validation.

#### Evaluation metrics

- **Overall:** MAE and RMSE on original scale (`expm1` predictions)
- **Stratified:** Report separately on top-10% most volatile hours — this is where operational value lies
- **Classification:** Precision-recall AUC on spike hours

Any model must beat the seasonal naive baseline **on the top-decile volatile hours** to be considered useful.